<a href="https://colab.research.google.com/github/UlaStats/MSc-project-pipe-failure-prediction/blob/main/Data%20Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSc Project - Data Preparation

This notebook contains code for and MSc Project about predicting time-to-failure of water mains in Scotland. In particular, the code contains data preparation - cleaning and manipulation.

## Import Data and Packages

In this section, required packages are imported and data is imported from my Google Drive.

In [1]:
# import packages required

from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# mount Google Drive

drive.mount("/content/drive", force_remount = True)

Mounted at /content/drive


In [3]:
# import data

assets = pd.read_csv("/content/drive/MyDrive/MSc project/Assets.csv", encoding = "latin1")
bursts = pd.read_csv("/content/drive/MyDrive/MSc project/burst_data.csv", encoding = "latin1")
soil = pd.read_csv("/content/drive/MyDrive/MSc project/soil-pipe-matched.csv", encoding="utf-8-sig")
pressure_assets = pd.read_csv("/content/drive/MyDrive/MSc project/pressure_assets_filtered.csv", encoding = "utf-8-sig")
nodes_pressure = pd.read_csv("/content/drive/MyDrive/MSc project/nodes_pressure.csv", encoding = "utf-8-sig")

/tmp/ipykernel_2922/71930382.py:3: DtypeWarning: Columns (22,23,28) have mixed types. Specify dtype option on import or set low_memory=False.
  assets = pd.read_csv("/content/drive/MyDrive/MSc project/Assets.csv", encoding = "latin1")
/tmp/ipykernel_2922/71930382.py:5: DtypeWarning: Columns (85,86,87,92,94,95,96,97,98,99,104,106,107,108) have mixed types. Specify dtype option on import or set low_memory=False.
  soil = pd.read_csv("/content/drive/MyDrive/MSc project/soil-pipe-matched.csv", encoding="utf-8-sig")
/tmp/ipykernel_2922/71930382.py:6: DtypeWarning: Columns (3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  pressure_assets = pd.read_csv("/content/drive/MyDrive/MSc project/pressure_assets_filtered.csv", encoding = "utf-8-sig")


# Data Cleaning

In this section data containing information abouts assets, bursts and soil is cleaned, details follow below.

### Assets

It is desired to remove assets that are not of interest to the water networks maintenance planning team and duplicated assets.

In [ ]:
# removing assets that are not of interest to the team

assets_cleaned = assets[assets["Type"] == "Distribution"] # only keeping distribution mains

assets_cleaned = assets_cleaned[assets_cleaned["Operational.Status"] != "Abandoned"] # remove abondened assets

assets_cleaned = assets_cleaned[assets_cleaned["Operational.Status"] != "Removed"] # remove removed assets

assets_cleaned = assets_cleaned.drop_duplicates(subset = ["Asset.ID"]) # remove duplicates


Some of the assets have missing values for diamater, age, material or surface. It is examined how severe the issue is by looking at the % of the total length of assets.

In [ ]:
# examine % of total length of assets that has missing diamater, age, material or surface code

print(assets_cleaned[pd.isnull(assets_cleaned['ARM_DIAMETER'])]["ARM_LENGTH"].sum() / assets_cleaned['ARM_LENGTH'].sum() * 100)

print(assets_cleaned[pd.isnull(assets_cleaned['ARM_AGE'])]["ARM_LENGTH"].sum() / assets_cleaned['ARM_LENGTH'].sum() * 100)

print(assets_cleaned[assets_cleaned['ARM_MATERIAL'] == "MISSING"]["ARM_LENGTH"].sum() / assets_cleaned['ARM_LENGTH'].sum() * 100)

print(assets_cleaned[assets_cleaned['ARM_SURFACE_CODE'] == "MISSING"]["ARM_LENGTH"].sum() / assets_cleaned['ARM_LENGTH'].sum() * 100)

Some missing values for the material could be imputed based on the information about the age of the asset. The age profile of assets by material is plotted and examined for this purpose.

In [ ]:
# plot age profile of assets by material

plt.figure(figsize = [15, 8])
sns.histplot(data = assets_cleaned,
             x = "ARM_AGE",
             hue = "ARM_MATERIAL")

# comment: if an asset it 8 years or younger, then it can be assumed it is made of PE - use for subsitituing material

Records containing missing values for diamater are removed, since they only make up less than 1% of the total length of all assets.

Records containing missing values for age are also removed, since they only make 5% of the total length. It is impossible to impute the data based on some other characteristics.

Records that had missing material and were younger than 8 years old, were imputer with PE as material. The remaining records with misising material are removed.

Assets with missing surface are removed from the data. It is impossible to impute this information using other characteristics.



In [ ]:
# missing values removal

assets_cleaned = assets_cleaned[pd.notnull(assets_cleaned["ARM_DIAMETER"])] # deletes records with missing diameter

assets_cleaned = assets_cleaned[pd.notnull(assets_cleaned["ARM_AGE"])] # deletes records with missing age

assets_cleaned.loc[(assets_cleaned["ARM_MATERIAL"] == "MISSING") & (assets_cleaned["ARM_AGE"] <= 8.0), "ARM_MATERIAL"] = "PE" # imputes assets that have missing material and are 8 or less years with PE

assets_cleaned = assets_cleaned[(assets_cleaned["ARM_MATERIAL"] != "MISSING")] # removes remaining assets with missing material

assets_cleaned = assets_cleaned[assets_cleaned["ARM_SURFACE_CODE"] != "Unknown"] # remove assets with unknown surface code (only 6 has unknown surface)

assets_cleaned = assets_cleaned[assets_cleaned["ARM_SURFACE_CODE"] != "MISSING"] # remove assets with missing surface code (~52k has missing surface)



Data is filtered to only contain columns of interest for further analysis. The selected columns are renamed.

In [ ]:
# select columns and re-name columns

assets_cleaned = assets_cleaned[["Asset.ID",
                                 "ARM_MATERIAL",
                                 "ARM_DIAMETER",
                                 "ARM_LENGTH",
                                 "ARM_LINING",
                                 "ARM_SURFACE_CODE",
                                 "ARM_COMMISSIONED"
                                 ]] # selecting attributes

assets_cleaned = assets_cleaned.rename(columns = {"ARM_MATERIAL":"Material",
                                                        "ARM_DIAMETER": "Diameter",
                                                        "ARM_LENGTH": "Length",
                                                        "ARM_LINING": "Lining",
                                                  "ARM_SURFACE_CODE": "Surface",
                                                  "ARM_COMMISSIONED": "Date_commissioned"}) # renaming columns


Data types of asset IT and date comissioned need to be changed.

In [ ]:
# change type

assets_cleaned['Asset.ID'] = assets_cleaned['Asset.ID'].astype(int) # changing asset type from float to integer

assets_cleaned['Date_commissioned'] = pd.to_datetime(assets_cleaned['Date_commissioned'], errors = "coerce", dayfirst=True)


Some assets don't have the commissioned date in the right format (most likely due to data entry error), and are therefore removed from the data set.

In [ ]:
# remove assets that don't have commissioned date in the right format

assets_cleaned = assets_cleaned[~(pd.isnull(assets_cleaned['Date_commissioned']))]

### Bursts

Bursts data is filtered to only contain relevant columns. Bursts without asset ID are also removed. Data types of some columns is also changed.

In [ ]:
# bursts cleaning

bursts_cleaned = bursts[["Asset.ID", "Raised.Date"]] # only selecting relevant information

bursts_cleaned = bursts_cleaned[pd.notnull(bursts_cleaned['Asset.ID'])] # removing bursts without asset ID

bursts_cleaned['Asset.ID'] = bursts_cleaned['Asset.ID'].astype(int) # changing Asset ID from float to integer

bursts_cleaned["Raised.Date"] = pd.to_datetime(bursts_cleaned["Raised.Date"], dayfirst=True) # transforms burst date to date format

Bursts that occured at the same asset on the same day, or next day, are removed from the records. They are assumed to come from the same event, but work orders were raised multiple times, hence the burst was logged multiple times.

In [ ]:
# remove bursts that were reported on the same day for the same asset

bursts_cleaned = bursts_cleaned.drop_duplicates(subset = ["Asset.ID", "Raised.Date"]) # delete duplicate bursts (same day, same asset)

In [ ]:
# only keep bursts if they occured more than 1 day between each other

bursts_cleaned = bursts_cleaned.sort_values(["Asset.ID", "Raised.Date"]) # sort bursts by date for each asset (required for the loop)


kept_rows = [] # create an empty set to record temaining bursts

for asset_id, bursts in bursts_cleaned.groupby("Asset.ID"): # note: this is a special type of loop that loops through the created groups

    previous_date = None

    for _, row in bursts.iterrows():

        current_date = row["Raised.Date"]

        if previous_date is None:
            kept_rows.append(row)


        else:
            day_diff = (current_date - previous_date).days

            if day_diff > 1:
                kept_rows.append(row)
                previous_date = current_date

bursts_cleaned = pd.DataFrame(kept_rows)




### Soil

Soil data is filtered to only contain relevant columns, instances with missing soil information are removed from the dataset and columns are re-named.

In [ ]:
# soil cleaning

soil_cleaned = soil[["asset_id", "msg84_2", "pH_900mm"]] # selecting only relevant information

soil_cleaned = soil_cleaned[pd.notnull(soil_cleaned["msg84_2"])] # removing assets with missing soil

soil_cleaned.columns = ["Asset.ID", "Soil", "Soil_pH"] # rename columns

### Pressure

Two data sets are available that contain information on pressure: one with asset ID and the corresponding nodes and the second one with the nodes and the corresponding pressure. Nodes are where the pressure is measured. Some initial cleaning is done on both datasets.

In [ ]:
# data cleaning of assets and pressure

pressure_assets['asset_id'] = pd.to_numeric(pressure_assets['asset_id'], errors = "coerce")

pressure_assets_cleaned = pressure_assets[pressure_assets['asset_id'].isin(assets_cleaned['Asset.ID'])] # filter for relevant assets only

pressure_assets_cleaned = pressure_assets_cleaned[["us_node_id", "ds_node_id", "asset_id"]] # keeping only relevant columns

In [ ]:
# data cleaning of nodes and pressure

pressure_nodes_cleaned = nodes_pressure[nodes_pressure['Node_Type'] == "Node"] # only pressure measured at node type = node is of interest

# below a list of upsteam and downstream nodes for each asset is created

ds_node_id = pressure_assets_cleaned['ds_node_id'].to_list()

us_node_id = pressure_assets_cleaned['us_node_id'].to_list()

ds_node_id.extend(us_node_id)

# the created list is used below to filter the pressure and nodes data to only contain relevant nodes

pressure_nodes_cleaned = pressure_nodes_cleaned[pressure_nodes_cleaned['node_id'].isin(ds_node_id)]

pressure_nodes_cleaned = pressure_nodes_cleaned[['node_id', "P_Max__m_"]] # only keeping relevant columns



In [ ]:
#pressure_nodes_cleaned
node_pressure = dict(zip(pressure_nodes_cleaned.node_id, pressure_nodes_cleaned.P_Max__m_))
node_pressure

Also, the maximum pressure from upstream and downstread node is extracted for each asset and the maximum of the two is selected as the final pressure reading.

The resultant data is cleaned to remove unnecessary columns and rename the remaining ones.

In [ ]:
def merge_pressure(x):
    us_pressure = node_pressure.get(x['us_node_id'])
    ds_pressure = node_pressure.get(x['ds_node_id'])
    #print (x)
    #print (us_pressure)
    #print (ds_pressure)
    if us_pressure is None:
      us_pressure = 0.0
    if ds_pressure is None:
      ds_pressure = 0.0

    asset_pressure = max(us_pressure, ds_pressure)
    return asset_pressure

pressure_assets_cleaned['pressure'] = pressure_assets_cleaned.apply(merge_pressure, axis=1)

In [ ]:
pressure_assets_cleaned

,us_node_id,ds_node_id,asset_id,pressure
0,000 430 01,999 429 02,3537609.0,42.93
1,000 430 02,000 430 01,3479220.0,42.93
2,000 436 01,000 430 02,3479220.0,50.24
3,000 438 01,000 436 01,3479220.0,54.13
4,000 440 01,000 438 01,3479220.0,54.13
...,...,...,...,...
772534,02586810661351_1,02586810661351_2,673860112.0,0.00
772536,02586820660739_1,02586820660740_1,528506.0,42.19
772538,02586820660740_1,02586820660740_3,694427775.0,42.19
772541,02586820660740_3,02586820660740_2,694427775.0,42.19


In [ ]:
# joining assets data with data on nodes

pre


pressure_assets_nodes = pd.merge(pressure_assets_cleaned, pressure_nodes_cleaned, how = "left", left_on = "us_node_id", right_on = "node_id")

pressure_assets_nodes = pd.merge(pressure_assets_nodes, pressure_nodes_cleaned, how = "left", left_on = "ds_node_id", right_on = "node_id")

# note : many-to

In [ ]:
# cretaing a pressure column (maximum pressure from upstream and downstream node)

pressure_assets_nodes["Pressure"] = pressure_assets_nodes[['P_Max__m__x', "P_Max__m__y"]].max().max()

In [ ]:
# cleaning the resultant dataset

pressure_assets_nodes = pressure_assets_nodes[["asset_id", "Pressure"]] # selecting only relevant columns

pressure_assets_nodes.columns = ["Asset.ID", "Pressure"] # renaming columns

pressure_assets_nodes['Asset.ID'] = pd.to_numeric(pressure_assets_nodes['Asset.ID']) # change type to integer

In [ ]:
pressure_assets_nodes

# Data manipulation

In this section, data is manipulated to join assets, bursts, pressure and soil data together. Transfromations are applied on the resultant dataset to create data sets in tidy data fromat with attributes suitable for modelling time-to-failure, probability of failure and failure rate.

### Joining

Data on assets needs to be joined up with information on bursts and soil to form one dataset for modelling purposes.

In [ ]:
# merging assets, bursts and soil together

assets_bursts = pd.merge(assets_cleaned, bursts_cleaned, how = "left", on = "Asset.ID")

assets_bursts_soil = pd.merge(assets_bursts, soil_cleaned, how = "left", on = "Asset.ID")

assets_bursts_soil = assets_bursts_soil[pd.notnull(assets_bursts_soil['Soil'])] # removing assets without soil

censored_data = assets_bursts_soil[~pd.notnull(assets_bursts_soil["Raised.Date"])] # keeping assets without bursts as censored data

assets_bursts_soil = assets_bursts_soil[pd.notnull(assets_bursts_soil['Raised.Date'])] # removing assets without bursts

Any assets that have censored data (have not had a burst yet) are saved in a separate file to later evaluate how the model performs on censored data.

In [ ]:
# save censored data

censored_data = assets_bursts_soil[~pd.notnull(assets_bursts_soil["Raised.Date"])] # keeping assets without bursts as censored data

censored_data = censored_data.drop(columns = ["Raised.Date"]) # removed raised date, as won't be needed for further analysis

censored_data.to_csv("/content/drive/MyDrive/MSc project/censored_data.csv", encoding ='latin1', index=False)

### Transformations

The dataset conatinin infromation on assets and correspondsing bursts needs to be transformed to a tidy format suitable for modelling.

The latest two failures for each asset need to be extracted to create an attributes with time-to-failure.

In [ ]:
# get index for latest two bursts for each asset

index = assets_bursts_soil.groupby("Asset.ID")["Raised.Date"].nlargest(2).reset_index(level = 0, drop = True).index

In [ ]:
# filter data to contain only latest two bursts

assets_bursts_soil_filtered = assets_bursts_soil.loc[index]

A column "Previous bursts" is added as attributes. This column corresponds to the number of previous failures.

In [ ]:
# Calculate previous fails for each asset

previous_bursts = assets_bursts_soil["Asset.ID"].value_counts() - 1

previous_bursts = previous_bursts.reset_index()


In [ ]:
# Add a new column

assets_bursts_soil_filtered = assets_bursts_soil_filtered.merge(previous_bursts, on = "Asset.ID", how = "left")
assets_bursts_soil_filtered = assets_bursts_soil_filtered.rename(columns={"count":"Previous bursts"}) # change column name

A "Time-to-break" column is added as a new attribute to the assets, bursts and soil data set. This column corresponds to the response attribute. Time-to-burst is calculated as time since install date or latest burst.

In [ ]:
# Change data type to date (necessary for calculating time difference)

assets_bursts_soil_filtered["Raised.Date"] = pd.to_datetime(assets_bursts_soil_filtered["Raised.Date"], dayfirst=True) # transofrms burst date to date format

assets_bursts_soil_filtered['Date_comm'] = pd.to_datetime(assets_bursts_soil_filtered["Date_commissioned"], errors = "coerce")

In [ ]:
# Sort records by raised date for each asset ID

assets_bursts_soil_filtered = assets_bursts_soil_filtered.sort_values(by = ["Asset.ID", "Raised.Date"])

In [ ]:
# Crate "Time-to-break" column

assets_bursts_soil_filtered.loc[assets_bursts_soil_filtered['Previous bursts'] == 0, "Time-to-break"] = \
  assets_bursts_soil_filtered['Raised.Date'] - assets_bursts_soil_filtered['Date_comm']

assets_bursts_soil_filtered.loc[assets_bursts_soil_filtered['Previous bursts'] > 0, "Time-to-break"] = \
  assets_bursts_soil_filtered.groupby("Asset.ID")["Raised.Date"].diff(1)

In [ ]:
# calculate "time-to-break"

# assets_bursts_soil_filtered["Time-to-break"] = assets_bursts_soil_filtered["Raised.Date"] - assets_bursts_soil_filtered['Date_comm']


In [ ]:
# Convert "Time-to-break" to days

assets_bursts_soil_filtered['Time-to-break'] = assets_bursts_soil_filtered['Time-to-break'].dt.days

In [ ]:
# remove assets that had fail on the same day as they were comissioned (likely an error)

assets_bursts_soil_filtered = assets_bursts_soil_filtered[~(assets_bursts_soil_filtered['Time-to-break'] == 0)]

A column "Age" is added. It equals 0 for assets that have not had any previous bursts, since time-to-failure since install is modelled. It equals to age at previous break, since time-to-failure since previous break is fitted.

In [ ]:
# Add "Age" column

assets_bursts_soil_filtered.loc[assets_bursts_soil_filtered["Previous bursts"] == 0, "Age"] = \
  (assets_bursts_soil)


# age at install

assets_bursts_soil_filtered.loc[assets_bursts_soil_filtered["Previous bursts"] > 0, "Age"] = \
  (assets_bursts_soil_filtered['Raised.Date'].shift(1) - assets_bursts_soil_filtered['Date_comm']).dt.days


In [ ]:
assets_bursts_soil_filtered = assets_bursts_soil_filtered[~(assets_bursts_soil_filtered['Age'] < 0)]

Save the final data set as attributes.

In [ ]:
# Only keep latest burst for each asset

index = assets_bursts_soil_filtered.groupby("Asset.ID")["Raised.Date"].nlargest(1).reset_index(level = 0, drop = True).index

In [ ]:
assets_bursts_soil_filtered = assets_bursts_soil_filtered.loc[index]

In [ ]:
# Remove assets that had burst data before commissioned date (happens due to data entry error)

assets_bursts_soil_filtered = assets_bursts_soil_filtered[assets_bursts_soil_filtered['Time-to-break'] > 0]

In [ ]:
# remove columns that won't be needed for further analysis

attributes = assets_bursts_soil.drop(columns = ["Date_comm", "Raised.Date"])

attributes = assets_bursts_soil_filtered

In [ ]:
# save attributes

attributes.to_csv("/content/drive/MyDrive/MSc project/attributes.csv", encoding ='latin1', index=False)